<a href="https://colab.research.google.com/github/choROPeNt/improved-diffusion/blob/dev/attention_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

In [2]:
print(torch.cuda.is_available())
print(torch.__version__)

True
2.9.0+cu126


In [3]:
def conv_nd(dims, *args, **kwargs):
    """
    Create a 1D, 2D, or 3D convolution module.
    """
    if dims == 1:
        return nn.Conv1d(*args, **kwargs)
    elif dims == 2:
        return nn.Conv2d(*args, **kwargs)
    elif dims == 3:
        return nn.Conv3d(*args, **kwargs)
    raise ValueError(f"unsupported dimensions: {dims}")

def zero_module(module):
    """
    Zero out the parameters of a module and return it.
    """
    for p in module.parameters():
        p.detach().zero_()
    return module

def normalization(channels):
    """
    Make a standard normalization layer.

    :param channels: number of input channels.
    :return: an nn.Module for normalization.
    """
    return GroupNorm32(32, channels)

class GroupNorm32(nn.GroupNorm):
    def forward(self, x):
        return super().forward(x.float()).type(x.dtype)

def checkpoint(func, inputs, params, flag):
    """
    Evaluate a function without caching intermediate activations, allowing for
    reduced memory at the expense of extra compute in the backward pass.

    :param func: the function to evaluate.
    :param inputs: the argument sequence to pass to `func`.
    :param params: a sequence of parameters `func` depends on but does not
                   explicitly take as arguments.
    :param flag: if False, disable gradient checkpointing.
    """
    if flag:
        args = tuple(inputs) + tuple(params)
        return CheckpointFunction.apply(func, len(inputs), *args)
    else:
        return func(*inputs)

class CheckpointFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, run_function, length, *args):
        ctx.run_function = run_function
        ctx.input_tensors = list(args[:length])
        ctx.input_params = list(args[length:])
        with torch.no_grad():
            output_tensors = ctx.run_function(*ctx.input_tensors)
        return output_tensors

    @staticmethod
    def backward(ctx, *output_grads):
        ctx.input_tensors = [x.detach().requires_grad_(True) for x in ctx.input_tensors]
        with torch.enable_grad():
            # Fixes a bug where the first op in run_function modifies the
            # Tensor storage in place, which is not allowed for detach()'d
            # Tensors.
            shallow_copies = [x.view_as(x) for x in ctx.input_tensors]
            output_tensors = ctx.run_function(*shallow_copies)
        input_grads = torch.autograd.grad(
            output_tensors,
            ctx.input_tensors + ctx.input_params,
            output_grads,
            allow_unused=True,
        )
        del ctx.input_tensors
        del ctx.input_params
        del output_tensors
        return (None, None) + input_grads

In [4]:
class AttentionBlock(nn.Module):
    """
    An attention block that allows spatial positions to attend to each other.

    Originally ported from here, but adapted to the N-d case.
    https://github.com/hojonathanho/diffusion/blob/1e0dceb3b3495bbe19116a5e1b3596cd0706c543/diffusion_tf/models/unet.py#L66.
    """

    def __init__(self, channels, num_heads=1, use_checkpoint=False):
        super().__init__()
        self.channels = channels
        self.num_heads = num_heads
        self.use_checkpoint = use_checkpoint

        self.norm = normalization(channels)
        self.qkv = conv_nd(1, channels, channels * 3, 1)
        self.attention = QKVAttention()
        self.proj_out = zero_module(conv_nd(1, channels, channels, 1))

    def forward(self, x):
        return checkpoint(self._forward, (x,), self.parameters(), self.use_checkpoint)

    def _forward(self, x):
        b, c, *spatial = x.shape
        x = x.reshape(b, c, -1)
        qkv = self.qkv(self.norm(x))
        qkv = qkv.reshape(b * self.num_heads, -1, qkv.shape[2])
        h = self.attention(qkv)
        h = h.reshape(b, -1, h.shape[-1])
        h = self.proj_out(h)
        return (x + h).reshape(b, c, *spatial)


class QKVAttention(nn.Module):
    """
    A module which performs QKV attention.
    """

    def forward(self, qkv):
        """
        Apply QKV attention.

        :param qkv: an [N x (C * 3) x T] tensor of Qs, Ks, and Vs.
        :return: an [N x C x T] tensor after attention.
        """
        ch = qkv.shape[1] // 3
        q, k, v = torch.split(qkv, ch, dim=1)
        scale = 1 / math.sqrt(math.sqrt(ch))
        weight = torch.einsum(
            "bct,bcs->bts", q * scale, k * scale
        )  # More stable with f16 than dividing afterwards
        weight = torch.softmax(weight.float(), dim=-1).type(weight.dtype)
        return torch.einsum("bts,bcs->bct", weight, v)

    @staticmethod
    def count_flops(model, _x, y):
        """
        A counter for the `thop` package to count the operations in an
        attention operation.

        Meant to be used like:

            macs, params = thop.profile(
                model,
                inputs=(inputs, timestamps),
                custom_ops={QKVAttention: QKVAttention.count_flops},
            )

        """
        b, c, *spatial = y[0].shape
        num_spatial = int(np.prod(spatial))
        # We perform two matmuls with the same number of ops.
        # The first computes the weight matrix, the second computes
        # the combination of the value vectors.
        matmul_ops = 2 * b * (num_spatial ** 2) * c
        model.total_ops += torch.DoubleTensor([matmul_ops])



class AttentionBlockSDPA(nn.Module):
    """
    Optimized attention block:
      - proper multi-head split
      - uses torch.nn.functional.scaled_dot_product_attention (fused kernels)
      - runs normalization in fp32 to avoid mixed-dtype issues (GroupNorm32 patterns)
    """
    def __init__(self, channels, num_heads=1, use_checkpoint=False):
        super().__init__()
        assert channels % num_heads == 0, "channels must be divisible by num_heads"
        self.channels = channels
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.use_checkpoint = use_checkpoint

        self.norm = normalization(channels)
        self.qkv = conv_nd(1, channels, channels * 3, 1)
        self.proj_out = zero_module(conv_nd(1, channels, channels, 1))

    def forward(self, x):
        if self.use_checkpoint:
            return checkpoint(self._forward, (x,), self.parameters(), True)
        return self._forward(x)

    def _forward(self, x):
        b, c, *spatial = x.shape
        t = 1
        for s in spatial:
            t *= s

        x_in = x
        x = x.reshape(b, c, t)  # [b, c, T]

        # Norm in fp32 for stability + to avoid GroupNorm mixed dtype issues
        x_norm = self.norm(x.float()).to(dtype=x.dtype)

        qkv = self.qkv(x_norm)          # [b, 3c, T]
        q, k, v = qkv.chunk(3, dim=1)   # each [b, c, T]

        # -> [b, heads, T, head_dim]
        q = q.reshape(b, self.num_heads, self.head_dim, t).transpose(2, 3)
        k = k.reshape(b, self.num_heads, self.head_dim, t).transpose(2, 3)
        v = v.reshape(b, self.num_heads, self.head_dim, t).transpose(2, 3)

        # Fused attention (Flash/mem-efficient/math backend chosen automatically)
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=False)

        # back to [b, c, T]
        out = out.transpose(2, 3).reshape(b, c, t)
        out = self.proj_out(out)

        # residual and reshape back
        out = (x + out).reshape(b, c, *spatial)
        return out


In [5]:
def test_attention_shapes(device="cuda"):
    torch.manual_seed(0)
    b, c, h, w = 2, 64, 32, 32

    blk = AttentionBlockSDPA(c, num_heads=4).to("cuda")   # keep params in fp32
    x = torch.randn(b, c, h, w, device="cuda", dtype=torch.float16, requires_grad=True)

    # sanity checks (these should both print cuda:0 and torch.float16)
    print("x:", x.device, x.dtype)
    print("norm weight:", blk.norm.weight.device, blk.norm.weight.dtype)


    with torch.autocast("cuda", dtype=torch.float16):
        y = blk(x)

    assert y.shape == x.shape
    y.square().mean().backward()
    assert x.grad is not None

In [6]:
test_attention_shapes()

x: cuda:0 torch.float16
norm weight: cuda:0 torch.float32


In [32]:
# --- Benchmarking utilities (time + peak memory) ---

def _sync_if_cuda(device: str):
    if device.startswith("cuda"):
        torch.cuda.synchronize()

@torch.no_grad()
def bench_forward(module: nn.Module, x: torch.Tensor, iters=50, warmup=10, use_autocast=True):
    device = str(x.device)
    module.eval()

    # Warmup
    for _ in range(warmup):
        if use_autocast and device.startswith("cuda"):
            with torch.autocast("cuda", dtype=x.dtype):
                _ = module(x)
        else:
            _ = module(x)
    _sync_if_cuda(device)

    # Measure peak memory (CUDA only)
    if device.startswith("cuda"):
        torch.cuda.reset_peak_memory_stats()

    t0 = time.perf_counter()
    for _ in range(iters):
        if use_autocast and device.startswith("cuda"):
            with torch.autocast("cuda", dtype=x.dtype):
                y = module(x)
        else:
            y = module(x)
    _sync_if_cuda(device)
    t1 = time.perf_counter()

    dt = (t1 - t0) / iters
    peak_mem_mb = None
    if device.startswith("cuda"):
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return dt, peak_mem_mb, y

def bench_forward_backward(module: nn.Module, x: torch.Tensor, iters=30, warmup=5, use_autocast=True):
    device = str(x.device)
    module.train()

    # Warmup
    for _ in range(warmup):
        x_ = x.detach().clone().requires_grad_(True)
        if use_autocast and device.startswith("cuda"):
            with torch.autocast("cuda", dtype=x.dtype):
                y = module(x_)
                loss = y.square().mean()
            loss.backward()
        else:
            y = module(x_)
            loss = y.square().mean()
            loss.backward()
    _sync_if_cuda(device)

    # Measure peak memory (CUDA only)
    if device.startswith("cuda"):
        torch.cuda.reset_peak_memory_stats()

    t0 = time.perf_counter()
    for _ in range(iters):
        x_ = x.detach().clone().requires_grad_(True)
        if use_autocast and device.startswith("cuda"):
            with torch.autocast("cuda", dtype=x.dtype):
                y = module(x_)
                loss = y.square().mean()
            loss.backward()
        else:
            y = module(x_)
            loss = y.square().mean()
            loss.backward()
    _sync_if_cuda(device)
    t1 = time.perf_counter()

    dt = (t1 - t0) / iters
    peak_mem_mb = None
    if device.startswith("cuda"):
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return dt, peak_mem_mb


def run_benchmarks(
    channels=128,
    num_heads=8,
    shape=(2, 128, 32, 32),
    dtype=torch.float16,
    device="cuda",
    iters_fwd=80,
    iters_bwd=30,
):
    b, c, *spatial = shape
    assert c == channels, "shape channels must match channels argument"

    # Input
    x = torch.randn(*shape, device=device, dtype=dtype)

    # Old (your original)
    old = AttentionBlock(channels, num_heads=num_heads, use_checkpoint=False).to(device)
    # Important: keep module fp32 and use autocast (avoids GroupNorm mixed dtype issues)
    # If you *must* use .half(), ensure your normalization casts params to float32.

    # New (SDPA)
    new = AttentionBlockSDPA(channels, num_heads=num_heads, use_checkpoint=False).to(device)

    # Forward-only
    old_dt, old_mem, _ = bench_forward(old, x, iters=iters_fwd, warmup=10, use_autocast=True)
    new_dt, new_mem, _ = bench_forward(new, x, iters=iters_fwd, warmup=10, use_autocast=True)

    # Forward+Backward
    old_dt_bwd, old_mem_bwd = bench_forward_backward(old, x, iters=iters_bwd, warmup=5, use_autocast=True)
    new_dt_bwd, new_mem_bwd = bench_forward_backward(new, x, iters=iters_bwd, warmup=5, use_autocast=True)

    # Print report
    def fmt_mem(m):
        return "n/a" if m is None else f"{m:8.1f} MiB"

    print("\n=== Attention Benchmark ===")
    print(f"device={device} dtype={dtype} shape={shape} heads={num_heads}")
    print("\nForward:")
    print(f"  old: {old_dt*1e3:8.3f} ms/iter  peak_mem={fmt_mem(old_mem)}")
    print(f"  new: {new_dt*1e3:8.3f} ms/iter  peak_mem={fmt_mem(new_mem)}")
    if old_dt > 0:
        print(f"  speedup: {old_dt/new_dt:6.2f}x")

    print("\nForward+Backward:")
    print(f"  old: {old_dt_bwd*1e3:8.3f} ms/iter  peak_mem={fmt_mem(old_mem_bwd)}")
    print(f"  new: {new_dt_bwd*1e3:8.3f} ms/iter  peak_mem={fmt_mem(new_mem_bwd)}")
    if old_dt_bwd > 0:
        print(f"  speedup: {old_dt_bwd/new_dt_bwd:6.2f}x, memory {new_mem_bwd/old_mem_bwd:.3f}")

    # Optional: verify shapes and finite
    with torch.no_grad():
        with torch.autocast("cuda", dtype=dtype) if device.startswith("cuda") else torch.no_grad():
            y_old = old(x)
            y_new = new(x)
        print("\nSanity:")
        print("  y_old:", tuple(y_old.shape), "finite:", torch.isfinite(y_old).all().item())
        print("  y_new:", tuple(y_new.shape), "finite:", torch.isfinite(y_new).all().item())



In [47]:
down = 3
spatial = 128

# Example usage:
run_benchmarks(
    channels=96*down,
    num_heads=4,
    shape=(2, 96*down, spatial//2**down, spatial//2**down, spatial//2**down),
    dtype=torch.float16,
    device="cuda",
)


=== Attention Benchmark ===
device=cuda dtype=torch.float16 shape=(2, 288, 16, 16, 16) heads=4

Forward:
  old:   14.998 ms/iter  peak_mem=  3435.9 MiB
  new:   28.373 ms/iter  peak_mem=  3353.0 MiB
  speedup:   0.53x

Forward+Backward:
  old:   52.179 ms/iter  peak_mem=  4744.2 MiB
  new:   66.418 ms/iter  peak_mem=  4247.0 MiB
  speedup:   0.79x, memory 0.895

Sanity:
  y_old: (2, 288, 16, 16, 16) finite: True
  y_new: (2, 288, 16, 16, 16) finite: True


In [18]:
def copy_attentionblock_weights(old_blk: nn.Module, new_blk: nn.Module):
    """Copy norm/qkv/proj weights so outputs are comparable."""
    new_blk.norm.load_state_dict(old_blk.norm.state_dict(), strict=True)
    new_blk.qkv.load_state_dict(old_blk.qkv.state_dict(), strict=True)
    new_blk.proj_out.load_state_dict(old_blk.proj_out.state_dict(), strict=True)

@torch.no_grad()
def check_equivalence(
    shape=(2, 64, 16, 16),
    device="cuda",
    dtype=torch.float32,
    num_heads=1,
    atol=None,
    rtol=None,
):
    """
    Compares old AttentionBlock vs new AttentionBlockSDPA with identical weights.
    Returns max_abs_diff and max_rel_diff (relative to |old|+eps).
    """
    b, c, *spatial = shape
    assert c % num_heads == 0

    old = AttentionBlock(c, num_heads=num_heads, use_checkpoint=False).to(device).eval()
    new = AttentionBlockSDPA(c, num_heads=num_heads, use_checkpoint=False).to(device).eval()
    copy_attentionblock_weights(old, new)

    x = torch.randn(*shape, device=device, dtype=dtype)

    # For fp16 checks, use autocast to mimic your training setup; for fp32 no autocast needed.
    if device.startswith("cuda") and dtype in (torch.float16, torch.bfloat16):
        with torch.autocast("cuda", dtype=dtype):
            y_old = old(x)
            y_new = new(x)
    else:
        y_old = old(x)
        y_new = new(x)

    diff = (y_old - y_new).abs()
    max_abs = diff.max().item()

    denom = y_old.abs().clamp_min(1e-8)
    max_rel = (diff / denom).max().item()

    # Choose tolerances if not given
    if atol is None or rtol is None:
        if dtype == torch.float32:
            atol = 1e-5 if atol is None else atol
            rtol = 1e-4 if rtol is None else rtol
        elif dtype == torch.bfloat16:
            atol = 2e-2 if atol is None else atol
            rtol = 5e-2 if rtol is None else rtol
        else:  # float16
            atol = 5e-2 if atol is None else atol
            rtol = 1e-1 if rtol is None else rtol

    ok = torch.allclose(y_old, y_new, atol=atol, rtol=rtol)

    print("\n=== Equivalence check ===")
    print(f"device={device} dtype={dtype} shape={shape} heads={num_heads}")
    print(f"allclose: {ok}  (atol={atol}, rtol={rtol})")
    print(f"max_abs_diff: {max_abs:.6g}")
    print(f"max_rel_diff: {max_rel:.6g}")

    return ok, max_abs, max_rel

In [19]:
# Recommended usage:
# 1) Check in float32 with heads=1 (should be very close, small numeric differences possible).
check_equivalence(shape=(2, 64, 128//2**3, 128//2**3), device="cuda", dtype=torch.float32, num_heads=1)
#
# 2) Then check in fp16/bf16 with looser tolerances:
check_equivalence(shape=(2, 64, 128//2**3, 128//2**3), device="cuda", dtype=torch.float16, num_heads=1)


=== Equivalence check ===
device=cuda dtype=torch.float32 shape=(2, 64, 16, 16) heads=1
allclose: True  (atol=1e-05, rtol=0.0001)
max_abs_diff: 0
max_rel_diff: 0

=== Equivalence check ===
device=cuda dtype=torch.float16 shape=(2, 64, 32, 32) heads=1
allclose: True  (atol=0.05, rtol=0.1)
max_abs_diff: 0
max_rel_diff: 0


(True, 0.0, 0.0)